# 05 - Merge tiled GeoTIFF exports

This notebook merges GeoTIFF tiles that belong to the same exported product.

It is useful when a Google Earth Engine export was split into multiple raster tiles because the file was too large to be downloaded as a single GeoTIFF.

Typical use cases:
- merge the tiles of `global_core_stats`
- merge the tiles of `global_hourly_core_stats`
- merge the tiles of `global_timeband_core_stats`
- merge the tiles of one or more `monthly_bundle_YYYYMM`

The notebook supports:
- **single-folder mode**: merge all TIFFs in one folder into one output file
- **batch mode**: scan subfolders and merge each tile group automatically


## 1. Install the required libraries

Run this only if the current environment does not already contain the required dependencies.


In [ ]:
# %pip install rasterio numpy


## 2. Import the libraries used by the merge workflow

These imports cover:
- raster reading and writing
- window management
- nodata handling
- folder scanning


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import math

import numpy as np
import rasterio
from rasterio.transform import Affine
from rasterio.windows import Window, from_bounds


## 3. Configure the merge task

Choose between:
- **single-folder mode**: all TIFFs in one folder belong to one merged output
- **batch mode**: each subfolder containing TIFFs is merged into its own output

Useful parameters:
- `nodata`: force an output nodata value, or keep `None` to auto-detect
- `compress`: `None` for faster write, `"LZW"` for smaller files
- `prefill_output_nodata`: slower but safer when overlap and non-trivial nodata coexist


In [ ]:
@dataclass
class MergeTilesConfig:
    # Input structure
    input_root: str = "data"
    output_root: str = "merged_outputs"
    single_folder_mode: bool = False

    # Single-folder mode only
    single_input_folder: str = "data/Global_core_stats"
    single_output_filename: str = "global_core_stats.tif"

    # Batch mode only
    folder_glob: str = "*"

    # Merge behaviour
    nodata: int | float | None = None
    read_all_bands_limit_mb: int = 2048
    gdal_cachemax_mb: int = 1024
    compress: str | None = None
    num_threads: str = "ALL_CPUS"
    prefill_output_nodata: bool = False

cfg = MergeTilesConfig()
cfg


## 4. Utility functions for validation and window management

The helpers below:
- compare nodata values safely
- detect overlap between windows
- validate that all source tiles share the same grid / CRS / datatype
- resolve a common nodata value when possible


In [ ]:
def _same_value(a, b) -> bool:
    if a is None and b is None:
        return True
    try:
        if np.isnan(a) and np.isnan(b):
            return True
    except TypeError:
        pass
    return a == b


def _nodata_mask(arr: np.ndarray, nodata):
    if nodata is None:
        return np.zeros(arr.shape, dtype=bool)
    try:
        if np.isnan(nodata):
            return np.isnan(arr)
    except TypeError:
        pass
    return arr == nodata


def _int_window(win: Window) -> Window:
    return Window(
        col_off=int(round(win.col_off)),
        row_off=int(round(win.row_off)),
        width=int(round(win.width)),
        height=int(round(win.height)),
    )


def _windows_intersect(a: Window, b: Window) -> bool:
    return not (
        a.col_off + a.width <= b.col_off
        or b.col_off + b.width <= a.col_off
        or a.row_off + a.height <= b.row_off
        or b.row_off + b.height <= a.row_off
    )


def _resolve_output_nodata(src_files, forced_nodata=None):
    if forced_nodata is not None:
        return forced_nodata

    first = src_files[0]
    ref_vals = tuple(first.nodatavals)

    if not ref_vals:
        return None

    if not all(_same_value(v, ref_vals[0]) for v in ref_vals):
        return None

    candidate = ref_vals[0]

    for src in src_files[1:]:
        vals = tuple(src.nodatavals)
        if len(vals) != len(ref_vals):
            return None
        if not all(_same_value(v, vals[0]) for v in vals):
            return None
        if not _same_value(vals[0], candidate):
            return None

    return candidate


def _validate_sources(src_files):
    ref = src_files[0]

    if ref.transform.b != 0 or ref.transform.d != 0:
        raise ValueError(f"{ref.name}: rotated / non north-up raster not supported.")

    for src in src_files[1:]:
        if src.crs != ref.crs:
            raise ValueError(f"Different CRS: {src.name}")
        if src.count != ref.count:
            raise ValueError(f"Different number of bands: {src.name}")
        if src.dtypes != ref.dtypes:
            raise ValueError(f"Different dtype: {src.name}")
        if src.res != ref.res:
            raise ValueError(f"Different resolution: {src.name}")
        if src.transform.b != 0 or src.transform.d != 0:
            raise ValueError(f"{src.name}: rotated / non north-up raster not supported.")

        raw_win = from_bounds(*src.bounds, transform=ref.transform)
        vals = np.array(
            [raw_win.col_off, raw_win.row_off, raw_win.width, raw_win.height],
            dtype=float,
        )
        if not np.allclose(vals, np.round(vals), atol=1e-6):
            raise ValueError(
                f"{src.name}: bounds are not aligned to the reference raster grid."
            )


## 5. Output metadata helpers

These functions:
- copy band descriptions and dataset tags from the reference tile
- choose a tiled output block size
- optionally prefill the destination raster with nodata


In [ ]:
def _copy_metadata(ref, dest):
    if ref.descriptions:
        for i, desc in enumerate(ref.descriptions, start=1):
            if desc:
                dest.set_band_description(i, desc)

    tags = ref.tags()
    if tags:
        dest.update_tags(**tags)


def _choose_blocksize(ref):
    if ref.is_tiled and ref.block_shapes:
        by, bx = ref.block_shapes[0]
        if 16 <= bx <= 4096 and 16 <= by <= 4096:
            return bx, by
    return 512, 512


def _prefill_output(dest, count, dtype, fill_value):
    print("Prefilling output with nodata...")
    fill_dtype = np.dtype(dtype)

    for _, win in dest.block_windows(1):
        arr = np.full(
            (count, int(win.height), int(win.width)),
            fill_value,
            dtype=fill_dtype,
        )
        dest.write(arr, window=win)


## 6. Band-wise merge strategies

Two merge strategies are implemented:

- **first valid wins**: used when a common output nodata exists
- **reverse overwrite**: used when no common nodata exists

In both cases, merging is done for **all bands**, not only band 1.


In [ ]:
def _merge_first_all_bands(dest, dst_window, new_data, src_nodatavals, out_nodata):
    old_data = dest.read(window=dst_window)
    merged = old_data.copy()

    for band_idx in range(new_data.shape[0]):
        src_nodata = src_nodatavals[band_idx]
        nd_for_new = src_nodata if src_nodata is not None else out_nodata

        new_valid = ~_nodata_mask(new_data[band_idx], nd_for_new)
        old_empty = _nodata_mask(old_data[band_idx], out_nodata)

        merged[band_idx] = np.where(
            new_valid & old_empty,
            new_data[band_idx],
            old_data[band_idx],
        )

    dest.write(merged, window=dst_window)


def _merge_reverse_overwrite_all_bands(dest, dst_window, new_data, src_nodatavals):
    old_data = dest.read(window=dst_window)
    merged = old_data.copy()

    for band_idx in range(new_data.shape[0]):
        src_nodata = src_nodatavals[band_idx]

        if src_nodata is None:
            merged[band_idx] = new_data[band_idx]
        else:
            new_valid = ~_nodata_mask(new_data[band_idx], src_nodata)
            merged[band_idx] = np.where(
                new_valid,
                new_data[band_idx],
                old_data[band_idx],
            )

    dest.write(merged, window=dst_window)


## 7. Core merge function

This is the main merge routine.

It:
- scans the input folder for TIFF tiles
- validates grid consistency
- computes the global output extent
- writes directly when there is no overlap
- switches to band-wise merge logic when overlap is present


In [ ]:
def merge_gee_tiles(
    input_folder: str,
    output_filepath: str,
    nodata=None,
    read_all_bands_limit_mb: int = 2048,
    gdal_cachemax_mb: int = 1024,
    compress: str | None = None,
    num_threads: str = "ALL_CPUS",
    prefill_output_nodata: bool = False,
):
    input_folder = Path(input_folder)
    output_filepath = Path(output_filepath)

    tif_files = sorted(input_folder.glob("*.tif"))
    print(f"Scanning TIFF files in: {input_folder}")

    if not tif_files:
        print("No TIFF files found.")
        return None

    print(f"Tiles found: {len(tif_files)}")
    src_files = [rasterio.open(fp) for fp in tif_files]

    try:
        _validate_sources(src_files)
        ref = src_files[0]

        out_nodata = _resolve_output_nodata(src_files, forced_nodata=nodata)
        if out_nodata is not None:
            print(f"Output nodata: {out_nodata}")
        else:
            print("No common output nodata found: using reverse-overwrite in overlap areas.")

        minx = min(src.bounds.left for src in src_files)
        miny = min(src.bounds.bottom for src in src_files)
        maxx = max(src.bounds.right for src in src_files)
        maxy = max(src.bounds.top for src in src_files)

        res_x, res_y = ref.res
        out_transform = Affine.translation(minx, maxy) * Affine.scale(res_x, -res_y)

        out_width = math.ceil((maxx - minx) / res_x)
        out_height = math.ceil((maxy - miny) / res_y)

        blockxsize, blockysize = _choose_blocksize(ref)

        out_meta = ref.profile.copy()
        out_meta.update(
            driver="GTiff",
            width=out_width,
            height=out_height,
            transform=out_transform,
            tiled=True,
            blockxsize=blockxsize,
            blockysize=blockysize,
            BIGTIFF="YES",
            SPARSE_OK=True,
            NUM_THREADS=num_threads,
        )

        if compress:
            out_meta["compress"] = compress
        if out_nodata is not None:
            out_meta["nodata"] = out_nodata

        print(f"Output size: {out_width} x {out_height} px")
        print(f"Block size: {blockxsize} x {blockysize}")
        print(f"GDAL cache: {gdal_cachemax_mb} MB")
        print(f"Compression: {compress if compress else 'none'}")
        print(f"Prefill nodata: {prefill_output_nodata}")

        with rasterio.Env(
            GDAL_CACHEMAX=gdal_cachemax_mb,
            GDAL_NUM_THREADS=num_threads,
        ):
            print(f"Creating output file: {output_filepath}")
            with rasterio.open(output_filepath, "w", **out_meta) as dest:
                _copy_metadata(ref, dest)
                if prefill_output_nodata and out_nodata is not None:
                    _prefill_output(dest, ref.count, ref.dtypes[0], out_nodata)

            with rasterio.open(output_filepath, "r+") as dest:
                if out_nodata is not None:
                    ordered_sources = src_files
                    strategy = "first_valid"
                else:
                    ordered_sources = list(reversed(src_files))
                    strategy = "reverse_overwrite"

                print(f"Merge strategy: {strategy}")
                written_windows = []

                for i, src in enumerate(ordered_sources, start=1):
                    print(f"Tile {i}/{len(ordered_sources)}: {Path(src.name).name}")

                    base_window = _int_window(
                        from_bounds(*src.bounds, transform=out_transform)
                        .round_offsets()
                        .round_lengths()
                    )

                    if base_window.width != src.width or base_window.height != src.height:
                        raise ValueError(
                            f"{src.name}: computed window ({base_window.width}x{base_window.height}) "
                            f"does not match source size ({src.width}x{src.height})."
                        )

                    overlaps_previous = any(
                        _windows_intersect(base_window, w) for w in written_windows
                    )

                    bytes_all_bands = (
                        src.width * src.height * src.count * np.dtype(src.dtypes[0]).itemsize
                    )
                    use_full_read = bytes_all_bands <= (read_all_bands_limit_mb * 1024 * 1024)

                    if not overlaps_previous:
                        print("  -> fast path: no previous overlap")
                        if use_full_read:
                            data = src.read()
                            dest.write(data, window=base_window)
                        else:
                            print("  -> block-based fallback for large tile")
                            for _, local_window in src.block_windows(1):
                                local_window = _int_window(local_window)
                                dst_window = Window(
                                    col_off=base_window.col_off + local_window.col_off,
                                    row_off=base_window.row_off + local_window.row_off,
                                    width=local_window.width,
                                    height=local_window.height,
                                )
                                data = src.read(window=local_window)
                                dest.write(data, window=dst_window)

                        written_windows.append(base_window)
                        continue

                    print("  -> overlap detected")

                    if use_full_read:
                        data = src.read()
                        if out_nodata is not None:
                            _merge_first_all_bands(
                                dest=dest,
                                dst_window=base_window,
                                new_data=data,
                                src_nodatavals=src.nodatavals,
                                out_nodata=out_nodata,
                            )
                        else:
                            _merge_reverse_overwrite_all_bands(
                                dest=dest,
                                dst_window=base_window,
                                new_data=data,
                                src_nodatavals=src.nodatavals,
                            )
                    else:
                        print("  -> overlap + large tile: merging by blocks")
                        for _, local_window in src.block_windows(1):
                            local_window = _int_window(local_window)
                            dst_window = Window(
                                col_off=base_window.col_off + local_window.col_off,
                                row_off=base_window.row_off + local_window.row_off,
                                width=local_window.width,
                                height=local_window.height,
                            )
                            data = src.read(window=local_window)

                            if out_nodata is not None:
                                _merge_first_all_bands(
                                    dest=dest,
                                    dst_window=dst_window,
                                    new_data=data,
                                    src_nodatavals=src.nodatavals,
                                    out_nodata=out_nodata,
                                )
                            else:
                                _merge_reverse_overwrite_all_bands(
                                    dest=dest,
                                    dst_window=dst_window,
                                    new_data=data,
                                    src_nodatavals=src.nodatavals,
                                )

                    written_windows.append(base_window)

        print("Merge completed successfully.")
        return output_filepath

    finally:
        for src in src_files:
            src.close()


## 8. Discover merge tasks

This helper builds the list of merge tasks from the current configuration.

In batch mode, every subfolder containing at least one TIFF becomes one merge job.


In [ ]:
def discover_merge_tasks(cfg: MergeTilesConfig) -> list[tuple[Path, Path]]:
    tasks: list[tuple[Path, Path]] = []

    if cfg.single_folder_mode:
        input_folder = Path(cfg.single_input_folder).resolve()
        output_path = Path(cfg.output_root).resolve() / cfg.single_output_filename
        tasks.append((input_folder, output_path))
        return tasks

    input_root = Path(cfg.input_root).resolve()
    output_root = Path(cfg.output_root).resolve()
    output_root.mkdir(parents=True, exist_ok=True)

    for folder in sorted(input_root.glob(cfg.folder_glob)):
        if not folder.is_dir():
            continue
        tif_files = list(folder.glob("*.tif"))
        if not tif_files:
            continue
        output_path = output_root / f"{folder.name}.tif"
        tasks.append((folder, output_path))

    return tasks


tasks = discover_merge_tasks(cfg)
print(f"Merge tasks found: {len(tasks)}")
tasks[:10]


## 9. Run the merge workflow

This cell executes all discovered merge tasks using the current configuration.


In [ ]:
results = []

for input_folder, output_path in tasks:
    print("\n" + "=" * 80)
    print(f"Input folder:  {input_folder}")
    print(f"Output file:   {output_path}")

    output_path.parent.mkdir(parents=True, exist_ok=True)

    merged = merge_gee_tiles(
        input_folder=str(input_folder),
        output_filepath=str(output_path),
        nodata=cfg.nodata,
        read_all_bands_limit_mb=cfg.read_all_bands_limit_mb,
        gdal_cachemax_mb=cfg.gdal_cachemax_mb,
        compress=cfg.compress,
        num_threads=cfg.num_threads,
        prefill_output_nodata=cfg.prefill_output_nodata,
    )
    results.append(merged)

print("\nAll merge tasks completed.")
results


## 10. Recommended usage notes

- Use **single-folder mode** if you are merging one export product at a time.
- Use **batch mode** if each product is stored in its own subfolder of tiles.
- If you know the real nodata value, set it explicitly.
- If overlap exists and nodata is tricky, consider `prefill_output_nodata=True`.


In [ ]:
print("Single-folder mode:", cfg.single_folder_mode)
print("Results written to:", Path(cfg.output_root).resolve())
